In [1]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from nibabel.processing import resample_to_output

from resources import find_tilt_error
from resources.cut_off_edges import cut_off_edges
from resources.find_rotation_error import find_rotation_error_by_profile
from resources.find_tilt_error import find_tilt_error_by_profile
from resources.rotate_image import rotate_volume
from resources.standarize_image import standardize_image

project_dir = Path.cwd().parents[1]

# Jeżeli notebook znajduje się np. w katalogu src:
if not (project_dir / "data").exists():
    project_dir = project_dir.parent

niftis_dir = project_dir / "data" / "niftis"
labels_path = niftis_dir / "train_labels_JNDlMjr.csv"

outputs_dir = project_dir / "src" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(labels_path)

target_column = "is_pathologic"

labels = labels_df.set_index("uid")[target_column]

nifti_paths = sorted(niftis_dir.glob("*.nii.gz"))

print("Katalog projektu:", project_dir)
print("Liczba obrazów:", len(nifti_paths))
print("Liczba etykiet:", len(labels))

data = np.load(
    project_dir / "data" / "processed_volumes_with_rotation.npz",
    allow_pickle=False,
)

X = data["X"].astype(np.float32)
y = data["y"].astype(np.int64)
uids = data["uid"]

from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
indices = np.arange(len(y))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

X_train = X[train_indices]
X_test = X[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

uids_train = uids[train_indices]
uids_test = uids[test_indices]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

class VolumeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        volume = torch.from_numpy(self.X[index]).float()
        label = torch.tensor(self.y[index], dtype=torch.float32)

        return volume, label

train_dataset = VolumeDataset(X_train, y_train)
test_dataset = VolumeDataset(X_test, y_test)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

validation_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

Katalog projektu: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons
Liczba obrazów: 1362
Liczba etykiet: 1362
X_train: (1089, 1, 50, 40, 35)
X_test: (273, 1, 50, 40, 35)


In [2]:
from torch import nn
class Simple3DCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=8,
                kernel_size=3,
                padding=1,
            ),
            nn.Mish(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=8,
                out_channels=16,
                kernel_size=3,
                padding=1,
            ),
            nn.Mish(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1,
            ),
            nn.Mish(),

            nn.AdaptiveAvgPool3d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x.squeeze(1)

In [14]:
# device = torch.device(
#     "cuda" if torch.cuda.is_available() else "cpu"
# )
#
# model = Simple3DCNN().to(device)
#
# criterion = nn.BCEWithLogitsLoss()
#
# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=0.00001,
#     weight_decay=0.0001,
# )
#
# print("Urządzenie:", device)
#
#
import copy
#
# best_validation_loss = float("inf")
# best_model_state = None

epochs = 2500

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for volumes_batch, labels_batch in train_loader:
        volumes_batch = volumes_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        logits = model(volumes_batch)
        loss = criterion(logits, labels_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(labels_batch)

    train_loss /= len(train_dataset)

    model.eval()
    validation_loss = 0.0

    with torch.no_grad():
        for volumes_batch, labels_batch in validation_loader:
            volumes_batch = volumes_batch.to(device)
            labels_batch = labels_batch.to(device)

            logits = model(volumes_batch)
            loss = criterion(logits, labels_batch)

            validation_loss += loss.item() * len(labels_batch)

    validation_loss /= len(test_dataset)

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoka {epoch + 1}/{epochs} | "
        f"train: {train_loss:.4f} | "
        f"validation: {validation_loss:.4f}"
    )

model.load_state_dict(best_model_state)

print("Najlepszy validation loss:", best_validation_loss)

Epoka 1/2500 | train: 0.4039 | validation: 0.3794
Epoka 2/2500 | train: 0.4062 | validation: 0.3828
Epoka 3/2500 | train: 0.3937 | validation: 0.3741
Epoka 4/2500 | train: 0.3986 | validation: 0.3730
Epoka 5/2500 | train: 0.3970 | validation: 0.3732
Epoka 6/2500 | train: 0.4007 | validation: 0.3738
Epoka 7/2500 | train: 0.3954 | validation: 0.3851
Epoka 8/2500 | train: 0.4012 | validation: 0.3824
Epoka 9/2500 | train: 0.3980 | validation: 0.3794


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
from datetime import datetime
import torch

models_dir = project_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

model_path = models_dir / f"swish_3d_cnn_{timestamp}_{best_validation_loss}.pth"

torch.save(
    model.state_dict(),
    model_path,
)

print(f"Zapisano model: {model_path} {best_validation_loss}")